# Урок 17 - Створення локальних AI агентів за допомогою Foundry Local та Qwen

У цьому зошиті ви створюєте **локального інженерного асистента**, який повністю працює на вашій робочій станції. Жодне хмарне виведення не використовується на жодному етапі. Асистент може:

1. **Викликати інструменти** через виклик функцій Qwen за допомогою Foundry Local.
2. **Переглядати та читати файли** в межах ізольованої директорії проекту.
3. **Аналізувати код** за допомогою простих метрик.
4. **Шукати документацію** за допомогою локального RAG (Chroma).
5. **Використовувати локальний сервер MCP** (якщо не налаштований, пропускається).

Код агента майже ідентичний до уроків у хмарі — змінюється лише клієнтський кінцевий пункт з хмари на `localhost`.


## Налаштування

Перед запуском цієї записної книжки:

1. **Встановіть Microsoft Foundry Local** (див. [документацію](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) для вашої ОС).
2. **Завантажте та запустіть модель Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Встановіть наведені нижче пакети Python.

Усе працює локально. Реалістичним мінімумом є машина з ~8 ГБ оперативної пам’яті.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Підключення до Foundry Local

`FoundryLocalManager` завантажує модель за потреби, запускає локальний сервіс і надає нам **сумісну з OpenAI кінцеву точку**. Потім ми вказуємо стандартному SDK OpenAI на неї. API-ключ є локальним маркером — жодні хмарні облікові дані не використовуються.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Локальні інструменти (пісочниця для файлових операцій)

Ми створюємо невеликий приклад проєкту на диску, а потім визначаємо інструменти, прив’язані до кореня цього проєкту. Перевірка пісочниці важлива навіть локально: інструмент, який читає довільні шляхи, працює з вашими користувацькими дозволами і може отримати доступ до будь-чого, що доступне вам.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Локальний RAG з Chroma

Ми вбудовуємо невелику кількість фрагментів документації в локальну колекцію Chroma. Chroma працює в процесі і зберігає вектори на диску — без сервера, без хмари. Інструмент `search_docs` витягує найбільш релевантні фрагменти для запиту.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Цикл виклику інструментів

Тепер ми реєструємо інструменти в моделі, використовуючи схему інструментів OpenAI, і запускаємо стандартний цикл виклику інструментів — модель запитує інструмент, ми виконуємо його локально, передаємо результат назад і повторюємо, поки модель не видасть кінцеву відповідь. Надійний виклик функцій Qwen — це те, що дозволяє це працювати на пристрої.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Локальний MCP (необов’язково)

MCP — це транспорт, а не хмарна служба — сервер MCP може працювати як локальний процес через `stdio`. Нижче наведена клітинка показує, як ви підключитесь до локального сервера MCP, якщо він у вас налаштований (наприклад, сервер файлової системи). Якщо `LOCAL_MCP_COMMAND` не встановлено, вона коректно пропускається, тож ноутбук усе одно виконується від початку до кінця без нього.

Примітка з безпеки: локальний сервер MCP працює з дозволами вашого користувача. Обмежте його зону дії директорією проєкту та перевіряйте його вивід перед тим, як діяти на його основі.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Резюме

Ви створили інженерного асистента, який повністю працює на вашому комп’ютері:

- **Foundry Local** працював з моделлю **Qwen** через сумісний з OpenAI інтерфейс — тому код агента збігається з уроками хмари.
- **Sandboxed tools** надали агенту доступ до файлів та аналіз коду без виходу за межі директорії проєкту.
- **Chroma** забезпечив **локальний RAG** по документації.
- **Local MCP** показав, як повторно використовувати екосистему MCP офлайн.

Жодних хмарних обчислень не було використано на жодному етапі.

### Виклик

Розширити локального агента, щоб:

1. **Працювати з кількома серверами MCP** — підключити файловий сервер і git-сервер і дозволити агенту обирати між ними.
2. **Використовувати локальну пам’ять** — зберігати коротку історію розмови на диск, щоб асистент пам’ятав попередні уточнення під час перезапусків ноутбука.
3. **Підтримувати локальне мультиагентське управління** — додати другого локального агента (наприклад, рецензента) і дозволити їм співпрацювати над завданням.

У наступному уроці ви дізнаєтесь, як захистити розгорнутих AI агентів.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
